# Trees, Hyperparameter Tuning, and Grid Search

Your Name

---

## Getting Started

- Replace **Your Name** above with your full name
- Automatic 0 if you include your student ID or any other identifying number
- Rename the file to `Your_Name.ipynb` before submitting
- When finished, share your Colab link with **Anyone with the link can view** privileges
- Paste the shared link into the Canvas submission

---

## How to Use This Notebook

This notebook is designed to be accessible to all users, including those navigating with a **screen reader**.

### Screen Reader Navigation Guide

Every code section in this notebook gives you a choice. Before each code cell you will find a navigation landmark with two options:

- **Skip this code cell** — Jumps past the code directly to the output explanation.
- **Go back and read the code cell** — Returns you to the top of the code section.

**Tips for screen reader users:**
- Press **H** to jump between major section headings
- Press **K** or **Tab** to move between links, including skip and return navigation links
- Press **D** to jump between landmark regions

---

## Learning Objectives

By the end of this assignment you will be able to:

1. Explain how a Decision Tree selects splits using Gini Impurity and Information Gain
2. Calculate Gini Impurity and Entropy by hand for a given node
3. Interpret a printed decision tree diagram — root node, splits, leaf nodes, and Gini values
4. Explain the Bias-Variance tradeoff and describe how each hyperparameter affects it
5. Implement Grid Search with cross-validation to tune a Random Forest
6. Apply SelectFromModel, RFE, and RFECV for feature selection
7. Compare Random Forest (bagging) to XGBoost and CatBoost (boosting)
8. Reason about whether a feature should be included in a model

---

## Notebook Roadmap

| Part | Topic |
| :--- | :--- |
| **1** | Decision Trees — theory and splitting criteria |
| **2** | Splitting intuition with the Wine dataset |
| **3** | Random Forests and hyperparameter tuning |
| **4** | Grid Search and the final model |
| **5** | Feature selection methods |
| **6** | Boosted trees on the Titanic dataset |
| **7** | Feature inclusion analysis |

---

# Part 1: Decision Trees

A decision tree is a decision support tool that uses a tree-like model of decisions and their possible consequences. It is one of the most interpretable machine learning models — you can trace exactly how the model made any individual prediction.

```
                          Which wine class is this?
                           Proline > 755 ?
                            (Root Node)
                                /\
                           yes /  \ no
                              /    \
                         class_0   continue splitting
                                        /\
                                   yes /  \ no
                                      /    \
                                  class_1  class_2
```

## Decision Tree Vocabulary

- **Root Node:** The first split — represents the entire training population
- **Decision Node:** Any internal node that splits into further sub-nodes
- **Splitting:** Dividing a node into two child nodes based on a feature threshold
- **Leaf / Terminal Node:** A node with no further splits — the final prediction
- **Branch:** A path from one node to a child node
- **Pruning:** Removing branches to reduce overfitting
- **Parent / Child:** The relationship between a node and the nodes it creates

---

## Splitting Criteria

Tree-based methods grow decision trees by repeatedly splitting data into purer subsets. A *splitting criterion* evaluates how good a candidate split is. The algorithm tries every possible feature and threshold, then selects the split that most reduces the criterion.

---

### Gini Impurity

Gini measures how often a randomly chosen sample would be **misclassified** if labeled according to the class distribution of that node.

$$\text{Gini} = 1 - \sum_{k=1}^{K} p_k^2$$

**Properties:**
- Lower Gini = purer node
- Gini = 0 means the node is perfectly pure (one class only)
- Maximum Gini for K classes = $1 - \frac{1}{K}$ (equal class distribution)
- For binary classification: maximum Gini = 0.5
- Used by default in CART (sklearn's `DecisionTreeClassifier`)
- Computationally efficient — no logarithms

---

### Entropy

Entropy comes from information theory. It measures the **uncertainty or disorder** in a node.

$$\text{Entropy} = -\sum_{k=1}^{K} p_k \log_2(p_k)$$

**Properties:**
- Entropy = 0 when the node is perfectly pure
- For binary classification: maximum Entropy = 1.0 (at 50/50 split)
- Penalizes impurity more heavily than Gini near balanced splits
- Used in ID3 and C4.5 algorithms

---

### Information Gain

Information Gain measures how much a split **reduces entropy** (or Gini) in the children.

$$\text{Information Gain} = H(\text{parent}) - \sum_{i} \frac{n_i}{n} H(\text{child}_i)$$

Where $H$ is entropy and $n_i$ is the number of samples in child $i$.

- **Higher Information Gain = better split**
- The algorithm selects the feature and threshold with the highest Information Gain
- Encourages splits that create homogeneous (pure) child nodes

---

### When to Use Each Criterion

| Criterion | Best When | Notes |
| :--- | :--- | :--- |
| **Gini** | Default, large datasets | Faster — no log computation |
| **Entropy** | Theoretically grounded work | Slightly prefers balanced splits |

In practice, the two criteria produce very similar trees. Start with Gini and switch to Entropy if you want to explore.

---

<a name="read-code-1"></a>

**Setup Cell — Install Packages and Import Libraries**

This cell installs XGBoost and CatBoost (needed for Part 6), then imports all libraries used throughout the notebook: pandas and numpy for data manipulation, matplotlib and seaborn for visualization, and a comprehensive set of scikit-learn modules for trees, ensembles, model selection, metrics, and feature selection.

<nav aria-label="Code cell 1 navigation">
<a href="#skip-code-1" aria-label="Skip code cell 1 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
import subprocess, sys
for pkg in ['xgboost', 'catboost']:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_wine
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.feature_selection import SelectFromModel, RFE, RFECV
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.pipeline import Pipeline

from xgboost import XGBClassifier
from catboost import CatBoostClassifier

np.random.seed(42)
%matplotlib inline
print('Setup complete.')

<a name="skip-code-1"></a>

**Expected output:** `Setup complete.` If xgboost or catboost installation fails, try `!pip install xgboost catboost` in a new cell.

<nav aria-label="Return navigation for code cell 1">
<a href="#read-code-1" aria-label="Go back and read code cell 1">&#8617; Go back and read the code cell</a>
</nav>

---
# Part 2: Splitting Intuition with the Wine Dataset

We will use the **Wine Recognition dataset** from scikit-learn. This dataset contains the results of a chemical analysis of wines grown in the same region of Italy by three different cultivars (wine producers). The task is to classify wines into three classes (class_0, class_1, class_2) based on 13 numeric chemical features including alcohol content, malic acid, ash, magnesium, flavanoids, and proline concentration.

This is an ideal teaching dataset for decision trees because:
- Three classes create a rich multi-class splitting problem
- Features have natural physical thresholds (alcohol percentage, proline levels)
- The classes are well-separated by a small number of features

---

<a name="read-code-2"></a>

**Cell 2 — Load the Wine Dataset and Fit a Full Decision Tree**

This cell loads the wine dataset (178 samples, 13 features, 3 classes), fits a DecisionTreeClassifier with Gini impurity and no depth limit, and plots the resulting tree. Because no constraints are applied, the tree will grow until every leaf is pure — useful for seeing the full structure before we discuss pruning.

<nav aria-label="Code cell 2 navigation">
<a href="#skip-code-2" aria-label="Skip code cell 2 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
wine = load_wine()
X = wine.data
y = wine.target

class_names = ['class_0 (Cultivar A)', 'class_1 (Cultivar B)', 'class_2 (Cultivar C)']

model_full = DecisionTreeClassifier(criterion='gini', random_state=42).fit(X, y)

plt.figure(figsize=(22, 10))
plot_tree(model_full,
          feature_names=wine.feature_names,
          class_names=class_names,
          filled=False,
          max_depth=3,
          fontsize=9)
plt.title('Wine Decision Tree (Gini, max_depth shown = 3 for readability)', fontsize=13)
plt.tight_layout()
plt.savefig('wine_tree_full.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Tree depth: {model_full.get_depth()}')
print(f'Leaf nodes: {model_full.get_n_leaves()}')

<a name="skip-code-2"></a>

**Expected output:** A decision tree diagram showing the first 3 levels and printed depth and leaf count. The full tree (all levels) will have many more leaves — this visual shows only the first 3 for readability. Notice how quickly the tree creates pure or near-pure nodes on the left branch.

**Accessibility note:** Add a Markdown cell below describing the tree structure — what feature appears at the root, and what the left and right branches represent.

<nav aria-label="Return navigation for code cell 2">
<a href="#read-code-2" aria-label="Go back and read code cell 2">&#8617; Go back and read the code cell</a>
</nav>

<a name="read-code-3"></a>

**Cell 3 — Train/Test Split and Pairplot**

This cell creates an 80/20 train/test split with stratification (to preserve the class distribution in both sets), then produces a pairplot colored by wine class. The pairplot reveals which feature pairs separate the three cultivars most cleanly.

<nav aria-label="Code cell 3 navigation">
<a href="#skip-code-3" aria-label="Skip code cell 3 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
df_wine = pd.DataFrame(data=X, columns=wine.feature_names)
df_wine['wine_class'] = pd.Categorical.from_codes(y, class_names)

X_train, X_test, y_train, y_test = train_test_split(
    df_wine.drop('wine_class', axis=1),
    df_wine['wine_class'],
    test_size=0.20, random_state=42, stratify=df_wine['wine_class']
)
print(f'Training set: {X_train.shape}')
print(f'Test set:     {X_test.shape}')

# Pairplot on a subset of the most informative features
plot_cols = ['alcohol', 'flavanoids', 'proline', 'color_intensity', 'wine_class']
sample = df_wine[plot_cols]

sns.pairplot(sample, hue='wine_class', palette=['#e74c3c','#2980b9','#27ae60'], plot_kws={'alpha': 0.6})
plt.suptitle('Wine Dataset — Pairplot of Key Features', y=1.02, fontsize=13)
plt.savefig('wine_pairplot.png', dpi=120, bbox_inches='tight')
plt.show()

<a name="skip-code-3"></a>

**Expected output:** Training shape (142, 13), test shape (36, 13), and a pairplot grid. Look for feature pairs where the three classes form clearly separated clusters. Alcohol vs Proline and Flavanoids vs Color_Intensity tend to show strong separation.

**Accessibility note:** Add a Markdown cell describing what you observe in the pairplot — which feature pairs separate the three cultivars most clearly?

<nav aria-label="Return navigation for code cell 3">
<a href="#read-code-3" aria-label="Go back and read code cell 3">&#8617; Go back and read the code cell</a>
</nav>

<a name="read-code-4"></a>

**Cell 4 — Feature Distributions as Histograms**

This cell plots histograms for all 13 wine features on the training set. Features with different distributions per class tend to be more useful for splitting.

<nav aria-label="Code cell 4 navigation">
<a href="#skip-code-4" aria-label="Skip code cell 4 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
X_train.hist(figsize=(16, 10), bins=20, color='steelblue', alpha=0.8, edgecolor='white')
plt.suptitle('Wine Training Set — Feature Distributions', fontsize=13)
plt.tight_layout()
plt.savefig('wine_histograms.png', dpi=150, bbox_inches='tight')
plt.show()

<a name="skip-code-4"></a>

**Expected output:** A 4×4 grid of histograms (one per feature). Note that some features like `proline` and `flavanoids` have multimodal distributions — multiple peaks suggest the feature takes on different typical values in different classes, making it highly useful for splitting.

**Accessibility note:** Add a Markdown cell identifying which features show multimodal distributions.

<nav aria-label="Return navigation for code cell 4">
<a href="#read-code-4" aria-label="Go back and read code cell 4">&#8617; Go back and read the code cell</a>
</nav>

<a name="read-code-5"></a>

**Cell 5 — Fit a Decision Tree on the Training Set and Plot**

This cell fits a Gini-based decision tree on the training data only (not the full dataset), then plots the tree limited to 3 levels for readability. Training on the split data rather than the full dataset is correct practice — we keep the test set unseen until final evaluation.

<nav aria-label="Code cell 5 navigation">
<a href="#skip-code-5" aria-label="Skip code cell 5 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
model_train = DecisionTreeClassifier(criterion='gini', random_state=42)
model_train.fit(X_train, y_train)

print(f'Training accuracy: {model_train.score(X_train, y_train):.3f}')
print(f'Test accuracy:     {model_train.score(X_test, y_test):.3f}')
print(f'Tree depth:        {model_train.get_depth()}')
print(f'Leaf nodes:        {model_train.get_n_leaves()}')

plt.figure(figsize=(22, 10))
plot_tree(model_train,
          feature_names=X_train.columns,
          class_names=class_names,
          filled=False,
          max_depth=3,
          fontsize=9)
plt.title('Wine Decision Tree — Trained on X_train (first 3 levels shown)', fontsize=13)
plt.tight_layout()
plt.savefig('wine_tree_trained.png', dpi=150, bbox_inches='tight')
plt.show()

<a name="skip-code-5"></a>

**Expected output:** Training accuracy typically near 1.00 (overfitting), test accuracy somewhat lower, and the tree diagram. An unconstrained tree will memorize the training data — this is the high-variance problem we will address with hyperparameters in Part 3.

<nav aria-label="Return navigation for code cell 5">
<a href="#read-code-5" aria-label="Go back and read code cell 5">&#8617; Go back and read the code cell</a>
</nav>

<a name="read-code-6"></a>

**Cell 6 — Calculate Gini Impurity at the Root Node**

This cell manually computes the Gini impurity at the root node using the class proportions in the training set. This gives you the starting impurity that the tree must reduce with each split.

<nav aria-label="Code cell 6 navigation">
<a href="#skip-code-6" aria-label="Skip code cell 6 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
import numpy as np

class_counts = y_train.value_counts(normalize=True)
print('Class proportions in training set:')
print(class_counts.round(4))

gini_root = 1 - np.sum(np.square(class_counts))
print(f'\nGini Impurity at Root Node: {gini_root:.4f}')

# For reference: maximum possible Gini for 3 classes = 1 - 1/3 = 0.6667
print(f'Maximum possible Gini for 3 classes: {1 - 1/3:.4f}')
print(f'Root node is {(1 - gini_root / (1 - 1/3)) * 100:.1f}% below maximum impurity')

<a name="skip-code-6"></a>

**Expected output:** Class proportions close to 1/3 each (wine is roughly balanced), root Gini close to 0.667, and a comparison to the theoretical maximum.

**Key insight:** The closer the root Gini is to the maximum, the more mixed the starting node is — and the more opportunity the tree has to reduce impurity through splitting. A balanced dataset like wine starts at near-maximum impurity, which gives the tree room to learn.

<nav aria-label="Return navigation for code cell 6">
<a href="#read-code-6" aria-label="Go back and read code cell 6">&#8617; Go back and read the code cell</a>
</nav>

## Reading the Decision Tree Diagram

Each node in the plotted tree shows:

- The **feature and threshold** used to split (e.g., `proline <= 755.0`)
- The **Gini impurity** at that node before the split
- The **number of samples** at that node
- The **class counts** `[class_0, class_1, class_2]`
- The **predicted class** (the majority class)

### Understanding Root Node Gini

With approximately equal numbers of the three wine classes, the root Gini is close to its maximum:

$$G = 1 - (p_0^2 + p_1^2 + p_2^2) \approx 1 - 3 \times \left(\frac{1}{3}\right)^2 = 1 - \frac{1}{3} \approx 0.667$$

### How Splits Are Chosen

At each node the algorithm:

1. Tries every possible feature and threshold combination
2. Computes the **weighted average Gini** of the resulting left and right children
3. Selects the split that **maximally reduces Gini** (maximizes the Gini decrease)

The weighted Gini after a split is:

$$\text{Weighted Gini} = \frac{n_L}{n} \cdot G_L + \frac{n_R}{n} \cdot G_R$$

The tree selects the split that minimizes this weighted Gini — equivalently, the split with the highest **Information Gain**.

### Why Certain Features Split First

Features that create the largest impurity reduction at the root appear at the top of the tree. For the wine dataset, `proline` (a measure of amino acid concentration) and `flavanoids` (polyphenol compounds) tend to dominate because they are highly discriminative across the three cultivars — the three classes have very different proline and flavanoid levels.

---

<a name="read-code-7"></a>

**Cell 7 — Scatter Plot Showing the Decision Boundaries**

This cell plots the two most informative features (flavanoids vs proline) colored by wine class, with the tree's decision boundary thresholds overlaid as lines. This makes the connection between the tree diagram and the actual data geometry concrete.

<nav aria-label="Code cell 7 navigation">
<a href="#skip-code-7" aria-label="Skip code cell 7 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
train_plot = X_train.copy()
train_plot['wine_class'] = y_train.values

# Get the first split threshold from the trained model
first_feature_idx = model_train.tree_.feature[0]
first_threshold   = model_train.tree_.threshold[0]
first_feature_name = X_train.columns[first_feature_idx]
print(f'Root split: {first_feature_name} <= {first_threshold:.2f}')

fig, ax = plt.subplots(figsize=(10, 7))
colors = {'class_0 (Cultivar A)': '#e74c3c', 'class_1 (Cultivar B)': '#2980b9', 'class_2 (Cultivar C)': '#27ae60'}
for cls, color in colors.items():
    mask = train_plot['wine_class'] == cls
    ax.scatter(train_plot.loc[mask, 'flavanoids'],
               train_plot.loc[mask, 'proline'],
               label=cls, color=color, alpha=0.7, edgecolors='white', s=60)

# Draw split lines
ax.axhline(y=755, color='black', linewidth=2, linestyle='--', label='Proline split (755)')
ax.axvline(x=1.575, color='gray', linewidth=1.5, linestyle=':', label='Flavanoids split (1.575)')

ax.set_xlabel('Flavanoids', fontsize=12)
ax.set_ylabel('Proline', fontsize=12)
ax.set_title('Wine Classes — Flavanoids vs Proline\n(with representative decision boundary lines)', fontsize=13)
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('wine_scatter_splits.png', dpi=150, bbox_inches='tight')
plt.show()

<a name="skip-code-7"></a>

**Expected output:** A scatter plot with three colored clusters and two boundary lines. The horizontal proline boundary should separate class_0 (high proline) from the others. The vertical flavanoids boundary should help separate class_1 from class_2.

**Accessibility note:** Add a Markdown cell describing the scatter plot: which class is above the proline line? Which class tends to have the highest flavanoid values?

<nav aria-label="Return navigation for code cell 7">
<a href="#read-code-7" aria-label="Go back and read code cell 7">&#8617; Go back and read the code cell</a>
</nav>

## Computing Information Gain for Feature Selection

The following cells implement Information Gain computation from scratch, so you can see exactly how the tree decides which feature to split on first.

The two functions below are:

- `compute_impurity(feature, criterion)` — calculates entropy or Gini for a series of labels
- `comp_feature_information_gain(df, target, feature, criterion)` — calculates the weighted impurity reduction from splitting on one feature

<a name="read-code-8"></a>

**Cell 8 — Define Impurity and Information Gain Functions**

This cell defines two functions. `compute_impurity` takes a pandas Series of class labels and computes either entropy or Gini. `comp_feature_information_gain` applies this to a full DataFrame: it computes the parent impurity, then computes the weighted child impurity for each level of the descriptive feature, and returns the Information Gain.

<nav aria-label="Code cell 8 navigation">
<a href="#skip-code-8" aria-label="Skip code cell 8 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
import numpy as np

def compute_impurity(feature, impurity_criterion):
    """
    Calculate impurity of a feature (pandas Series of class labels).
    Supported criteria: 'entropy', 'gini'
    """
    probs = feature.value_counts(normalize=True)

    if impurity_criterion == 'entropy':
        impurity = -1 * np.sum(np.log2(probs + 1e-9) * probs)  # 1e-9 avoids log(0)
    elif impurity_criterion == 'gini':
        impurity = 1 - np.sum(np.square(probs))
    else:
        raise ValueError('Unknown impurity criterion: use entropy or gini')
    return round(impurity, 4)


def comp_feature_information_gain(df, target, descriptive_feature, split_criterion):
    """
    Calculate information gain for splitting on a categorical descriptive feature.
    Supported criteria: 'entropy', 'gini'
    """
    target_entropy       = compute_impurity(df[target], split_criterion)
    entropy_list, weight_list = [], []

    for level in df[descriptive_feature].unique():
        df_level        = df[df[descriptive_feature] == level]
        entropy_level   = compute_impurity(df_level[target], split_criterion)
        entropy_list.append(entropy_level)
        weight_list.append(len(df_level) / len(df))

    remaining_impurity = np.sum(np.array(entropy_list) * np.array(weight_list))
    information_gain   = target_entropy - remaining_impurity

    print(f'  {descriptive_feature:30s}  IG = {information_gain:.4f}')
    return information_gain

print('Information gain functions defined.')

<a name="skip-code-8"></a>

**Expected output:** `Information gain functions defined.`

These functions work on categorical features. For continuous features (like the wine dataset's numeric columns), the decision tree algorithm must try every possible threshold — sklearn does this internally. These functions give you the intuition behind that process.

<nav aria-label="Return navigation for code cell 8">
<a href="#read-code-8" aria-label="Go back and read code cell 8">&#8617; Go back and read the code cell</a>
</nav>

<a name="read-code-9"></a>

**Cell 9 — Compare Entropy Information Gain Across Wine Features**

This cell bins each continuous wine feature into quintiles (5 equal-frequency bins) to make them categorical, then calculates the entropy-based Information Gain for each feature against the wine class target. Features are printed in descending order of Information Gain.

<nav aria-label="Code cell 9 navigation">
<a href="#skip-code-9" aria-label="Skip code cell 9 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
# Bin continuous features into quintiles for categorical IG calculation
df_ig = df_wine.copy()
for col in wine.feature_names:
    df_ig[col] = pd.qcut(df_ig[col], q=5, labels=False, duplicates='drop')
df_ig['wine_class'] = df_wine['wine_class']

print('=== Entropy-based Information Gain ===')
ig_entropy = {}
for feature in wine.feature_names:
    ig_entropy[feature] = comp_feature_information_gain(df_ig, 'wine_class', feature, 'entropy')

top_entropy = sorted(ig_entropy.items(), key=lambda x: x[1], reverse=True)
print('\nRanking (highest to lowest):')
for i, (feat, ig) in enumerate(top_entropy, 1):
    print(f'  {i:2d}. {feat:30s}  {ig:.4f}')

<a name="skip-code-9"></a>

**Expected output:** A ranked list of wine features by Information Gain (entropy). Expect `proline`, `flavanoids`, `color_intensity`, and `alcohol` near the top — these are the most discriminative features for wine classification. The top-ranked feature should match (approximately) what appears at the root of the decision tree.

<nav aria-label="Return navigation for code cell 9">
<a href="#read-code-9" aria-label="Go back and read code cell 9">&#8617; Go back and read the code cell</a>
</nav>

<a name="read-code-10"></a>

**Cell 10 — Compare Gini Information Gain Across Wine Features**

This cell runs the same calculation using Gini impurity instead of entropy. Compare the rankings with the entropy results from Cell 9 — they should be very similar, confirming that the choice between Gini and entropy rarely changes which feature is selected first.

<nav aria-label="Code cell 10 navigation">
<a href="#skip-code-10" aria-label="Skip code cell 10 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
print('=== Gini-based Information Gain ===')
ig_gini = {}
for feature in wine.feature_names:
    ig_gini[feature] = comp_feature_information_gain(df_ig, 'wine_class', feature, 'gini')

top_gini = sorted(ig_gini.items(), key=lambda x: x[1], reverse=True)
print('\nRanking (highest to lowest):')
for i, (feat, ig) in enumerate(top_gini, 1):
    print(f'  {i:2d}. {feat:30s}  {ig:.4f}')

# Compare top feature between criteria
print(f'\nEntropy top feature: {top_entropy[0][0]}')
print(f'Gini    top feature: {top_gini[0][0]}')
print('Same?', top_entropy[0][0] == top_gini[0][0])

<a name="skip-code-10"></a>

**Expected output:** Gini rankings that are very similar to the entropy rankings. The top feature should be the same or very close under both criteria. This confirms the practical guideline: when in doubt, default to Gini — it computes faster and produces nearly identical splits.

<nav aria-label="Return navigation for code cell 10">
<a href="#read-code-10" aria-label="Go back and read code cell 10">&#8617; Go back and read the code cell</a>
</nav>

---
# Part 3: Random Forests and Hyperparameter Tuning

A **Random Forest** is an ensemble of decision trees. Each tree is trained on a different bootstrap sample of the data and considers only a random subset of features at each split. The final prediction is the majority vote across all trees.

## Ensemble Learning

Ensemble methods combine multiple weak learners to produce a stronger learner. A single unconstrained decision tree has **high variance** — it fits the training data perfectly but generalizes poorly. A Random Forest **reduces variance** by averaging many such trees.

The key principle: even though each individual tree overfits, their errors are largely uncorrelated (because each tree sees different data). When you average uncorrelated errors, they cancel out.

## Parameters vs Hyperparameters

| Term | Definition | Who Sets It |
| :--- | :--- | :--- |
| **Parameter** | Values learned from data (e.g., split thresholds, node values) | The algorithm |
| **Hyperparameter** | Values that control the learning process | The data scientist |

## The Bias-Variance Tradeoff

Every model's test error decomposes into:

$$\text{Total Error} = \text{Bias}^2 + \text{Variance} + \text{Irreducible Noise}$$

| Source | Cause | Tree Symptom | Fix |
| :--- | :--- | :--- | :--- |
| **Bias** | Model too simple | Very shallow tree, underfits | Increase depth, add features |
| **Variance** | Model too complex | Deep tree, memorizes noise | Limit depth, use Random Forest |

### How Each Hyperparameter Affects the Tradeoff

**`max_depth`**
- High value → Low bias, High variance (deep tree can overfit)
- Low value → High bias, Low variance (shallow tree underfits)

**`min_samples_split`**
- Higher value → Forces broader splits → Higher bias, Lower variance
- The minimum samples required at a node before it is eligible to split

**`min_samples_leaf`**
- Higher value → Requires more samples in every leaf → Higher bias, Lower variance
- Prevents the tree from creating very specific rules for tiny data subsets

**`bootstrap`**
- `True` (default): Each tree trains on a different random sample with replacement
- This is called **Bagging** (Bootstrap Aggregating) — it is the primary mechanism
  by which Random Forest reduces variance
- `False`: Every tree sees the full training set — less diversity, more variance

**`criterion`** (gini vs entropy)
- Does not primarily affect bias/variance — affects which splits are chosen
- Both criteria lead to similar trees in practice

## Cross-Validation

**Cross-validation** is the gold standard for model evaluation because it uses every observation for both training and validation.

In **K-fold cross-validation**:
1. Split training data into K equally sized folds
2. Train K times — each time using K-1 folds for training and 1 fold for validation
3. Average the K validation scores

This gives a more reliable estimate of test performance than a single train/validation split. **Always perform cross-validation on training data only — never on the test set.**

## Grid Search

Grid Search is exhaustive hyperparameter optimization: it tries every combination of the hyperparameter values you specify and uses cross-validation to find the combination with the best average score.

**Important:** Never make adjustments to your model based on test set results. Grid Search should only ever see training data.

---

<a name="read-code-11"></a>

**Cell 11 — Decision Tree with Entropy and No Constraints**

This cell fits a decision tree using entropy as the splitting criterion with no depth or leaf constraints. The tree will grow until every leaf is pure — observe the training vs test accuracy gap, which signals overfitting.

<nav aria-label="Code cell 11 navigation">
<a href="#skip-code-11" aria-label="Skip code cell 11 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
from sklearn.metrics import accuracy_score
from sklearn import tree

hyperparameters_unconstrained = {'criterion': 'entropy'}

model_unconstrained = DecisionTreeClassifier(random_state=42).set_params(**hyperparameters_unconstrained)
model_unconstrained.fit(X_train, y_train)
preds_unconstrained = model_unconstrained.predict(X_test)

print('=== Unconstrained Decision Tree (entropy) ===')
print(f'Training accuracy: {model_unconstrained.score(X_train, y_train):.4f}')
print(f'Test accuracy:     {accuracy_score(y_test, preds_unconstrained):.4f}')
print(f'Tree depth:        {model_unconstrained.get_depth()}')
print(f'Leaf nodes:        {model_unconstrained.get_n_leaves()}')

plt.figure(figsize=(22, 10))
plot_tree(model_unconstrained, feature_names=X_train.columns,
          class_names=class_names, filled=False, max_depth=3, fontsize=9)
plt.title('Unconstrained Tree (entropy) — First 3 Levels')
plt.tight_layout()
plt.savefig('wine_tree_unconstrained.png', dpi=150, bbox_inches='tight')
plt.show()

<a name="skip-code-11"></a>

**Expected output:** Training accuracy of 1.000 (perfect) with test accuracy lower — this gap is the classic overfitting signature. The tree grew many levels deep to perfectly memorize every training example, including noise. This is the high-variance problem.

<nav aria-label="Return navigation for code cell 11">
<a href="#read-code-11" aria-label="Go back and read code cell 11">&#8617; Go back and read the code cell</a>
</nav>

<a name="read-code-12"></a>

**Cell 12 — Decision Tree with Constraints (Regularization)**

This cell applies two constraints: `max_depth=3` limits the tree to 3 levels, and `max_leaf_nodes=6` caps the number of leaves. These are forms of **regularization** that trade some bias for dramatically reduced variance. Observe the change in the train/test accuracy gap.

<nav aria-label="Code cell 12 navigation">
<a href="#skip-code-12" aria-label="Skip code cell 12 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
hyperparameters_constrained = {
    'criterion': 'entropy',
    'max_depth': 3,
    'max_leaf_nodes': 6
}

model_constrained = DecisionTreeClassifier(random_state=42).set_params(**hyperparameters_constrained)
model_constrained.fit(X_train, y_train)
preds_constrained = model_constrained.predict(X_test)

print('=== Constrained Decision Tree (max_depth=3, max_leaf_nodes=6) ===')
print(f'Training accuracy: {model_constrained.score(X_train, y_train):.4f}')
print(f'Test accuracy:     {accuracy_score(y_test, preds_constrained):.4f}')
print(f'Tree depth:        {model_constrained.get_depth()}')
print(f'Leaf nodes:        {model_constrained.get_n_leaves()}')

plt.figure(figsize=(14, 7))
plot_tree(model_constrained, feature_names=X_train.columns,
          class_names=class_names, filled=False, fontsize=9)
plt.title('Constrained Tree (max_depth=3, max_leaf_nodes=6)')
plt.tight_layout()
plt.savefig('wine_tree_constrained.png', dpi=150, bbox_inches='tight')
plt.show()

<a name="skip-code-12"></a>

**Expected output:** Training accuracy lower than 1.000 but test accuracy may be similar to or higher than the unconstrained tree. The train/test gap should be smaller — the constraints prevented the tree from memorizing noise. The tree diagram is now simple enough to read entirely.

<nav aria-label="Return navigation for code cell 12">
<a href="#read-code-12" aria-label="Go back and read code cell 12">&#8617; Go back and read the code cell</a>
</nav>

---
# Part 4: Grid Search and the Final Model

We will use Grid Search with 10-fold cross-validation to systematically find the best hyperparameter combination for a Random Forest on the wine dataset.

**Design your grid thoughtfully.** For each hyperparameter, choose a small number of values that span a meaningful range without creating an explosively large search space. A good rule of thumb: aim for under 50 total combinations.

---

<a name="read-code-13"></a>

**Cell 13 — Load Wine Data and Create Train/Test Split for Grid Search**

This cell creates a clean train/test split for the grid search section. We use 25% for the test set and stratify to preserve class proportions. The test set is held out completely — Grid Search will only ever see training data.

<nav aria-label="Code cell 13 navigation">
<a href="#skip-code-13" aria-label="Skip code cell 13 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
wine   = load_wine()
X_full = pd.DataFrame(wine.data, columns=wine.feature_names)
y_full = pd.Series(wine.target)

X_tr, X_te, y_tr, y_te = train_test_split(
    X_full, y_full, test_size=0.25, random_state=42, stratify=y_full
)
print(f'Training: {X_tr.shape}  |  Test: {X_te.shape}')
print(f'Class distribution (train): {dict(y_tr.value_counts().sort_index())}')

<a name="skip-code-13"></a>

**Expected output:** Training set of approximately 133 samples, test set of 45.

<nav aria-label="Return navigation for code cell 13">
<a href="#read-code-13" aria-label="Go back and read code cell 13">&#8617; Go back and read the code cell</a>
</nav>

<a name="read-code-14"></a>

**Cell 14 — Grid Search with Cross-Validation**

This cell runs GridSearchCV over a carefully chosen set of hyperparameter combinations. With `cv=10`, each combination is evaluated 10 times — once per fold — and the average accuracy is used to rank combinations. This may take 30–60 seconds to complete.

<nav aria-label="Code cell 14 navigation">
<a href="#skip-code-14" aria-label="Skip code cell 14 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
hyperparameter_grid = {
    'max_depth':        [2, 3, 4],
    'min_samples_split':[3, 5],
    'min_samples_leaf': [2, 4],
    'bootstrap':        [True, False],
    'criterion':        ['entropy', 'gini']
}

n_combinations = 1
for v in hyperparameter_grid.values():
    n_combinations *= len(v)
print(f'Total combinations to evaluate: {n_combinations} × 10 folds = {n_combinations * 10} fits')

grid_search = GridSearchCV(
    estimator  = RandomForestClassifier(n_estimators=100, random_state=42),
    param_grid = hyperparameter_grid,
    scoring    = 'accuracy',
    cv         = 10,
    n_jobs     = -1
)
grid_search.fit(X_tr, y_tr)

print(f'\nBest CV accuracy:  {grid_search.best_score_:.4f}')
print(f'Best parameters:')
for k, v in grid_search.best_params_.items():
    print(f'  {k:25s}: {v}')

<a name="skip-code-14"></a>

**Expected output:** Grid search results including the best cross-validation accuracy (typically above 0.95 for wine) and the best parameter combination. The cross-validation score is more reliable than a single train/validation split because it averages over 10 different validation folds.

<nav aria-label="Return navigation for code cell 14">
<a href="#read-code-14" aria-label="Go back and read code cell 14">&#8617; Go back and read the code cell</a>
</nav>

<a name="read-code-15"></a>

**Cell 15 — Apply Best Parameters and Evaluate on the Held-Out Test Set**

This cell applies the best parameters found by Grid Search to a new model, fits it on the full training set, and evaluates it on the test set. Note that we use `set_params(**best_parameters)` — this is a clean way to apply a parameter dictionary to any sklearn model.

<nav aria-label="Code cell 15 navigation">
<a href="#skip-code-15" aria-label="Skip code cell 15 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
best_params = grid_search.best_params_

rf_best = RandomForestClassifier(n_estimators=100, random_state=42).set_params(**best_params)
rf_best.fit(X_tr, y_tr)
preds_best = rf_best.predict(X_te)

print('=== Final Random Forest — Test Set Evaluation ===')
print(f'Test accuracy: {accuracy_score(y_te, preds_best):.4f}')
print(f'\nClassification Report:')
print(classification_report(y_te, preds_best, target_names=['Cultivar A', 'Cultivar B', 'Cultivar C']))

cm = confusion_matrix(y_te, preds_best)
fig, ax = plt.subplots(figsize=(7, 5))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Cultivar A', 'Cultivar B', 'Cultivar C'])
disp.plot(ax=ax, colorbar=False)
plt.title('Confusion Matrix — Random Forest (Best Parameters)')
plt.tight_layout()
plt.savefig('wine_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

<a name="skip-code-15"></a>

**Expected output:** Test accuracy typically above 0.95, a full classification report showing precision, recall, and F1 per class, and a confusion matrix showing where any misclassifications occur. For wine, most errors are between Cultivar B and Cultivar C (the two most similar classes).

**Accessibility note:** Add a Markdown cell describing the confusion matrix — how many correct predictions appear on the diagonal, and which classes are confused?

<nav aria-label="Return navigation for code cell 15">
<a href="#read-code-15" aria-label="Go back and read code cell 15">&#8617; Go back and read the code cell</a>
</nav>

---
# Part 5: Feature Selection

Feature selection reduces dimensionality, speeds up training, and often improves generalization by removing noise features. We will compare three methods:

| Method | Type | Strategy |
| :--- | :--- | :--- |
| **SelectFromModel** | Embedded | Keep features whose importance exceeds the mean |
| **RFE** | Wrapper | Iteratively remove the weakest feature and refit |
| **RFECV** | Wrapper + CV | RFE with cross-validation to find the optimal feature count |

---

<a name="read-code-16"></a>

**Cell 16 — SelectFromModel (Random Forest Importance Threshold)**

This cell fits a Random Forest and uses `SelectFromModel` to keep only the features whose importance score exceeds the mean importance across all features. This is a fast, embedded method — feature selection happens as part of model training.

<nav aria-label="Code cell 16 navigation">
<a href="#skip-code-16" aria-label="Skip code cell 16 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
selector_sfm = SelectFromModel(RandomForestClassifier(n_estimators=100, random_state=42))
selector_sfm.fit(X_tr, y_tr)

selected_sfm = X_tr.columns[selector_sfm.get_support()]
print('SelectFromModel — Selected Features:')
for i, f in enumerate(selected_sfm, 1):
    print(f'  {i}. {f}')
print(f'\n{len(selected_sfm)} of {X_tr.shape[1]} features selected')

# Evaluate model on selected features
rf_sfm = RandomForestClassifier(n_estimators=100, random_state=42)
rf_sfm.fit(X_tr[selected_sfm], y_tr)
print(f'Test accuracy with selected features: {rf_sfm.score(X_te[selected_sfm], y_te):.4f}')

<a name="skip-code-16"></a>

**Expected output:** A subset of the 13 wine features (typically 4–7) selected and a test accuracy with those features. If accuracy is similar to the full-feature model, the removed features were adding noise, not signal.

<nav aria-label="Return navigation for code cell 16">
<a href="#read-code-16" aria-label="Go back and read code cell 16">&#8617; Go back and read the code cell</a>
</nav>

<a name="read-code-17"></a>

**Cell 17 — Recursive Feature Elimination (RFE)**

RFE trains a Random Forest, ranks features by importance, removes the weakest feature, and repeats until the target number of features remains. We target 5 features — adjust `n_features_to_select` to explore.

<nav aria-label="Code cell 17 navigation">
<a href="#skip-code-17" aria-label="Skip code cell 17 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
selector_rfe = RFE(
    estimator=RandomForestClassifier(n_estimators=100, random_state=42),
    n_features_to_select=5
)
selector_rfe.fit(X_tr, y_tr)

selected_rfe = X_tr.columns[selector_rfe.get_support()]
print('RFE — Selected Features (target: 5):')
for i, f in enumerate(selected_rfe, 1):
    print(f'  {i}. {f}')

# Feature importances for selected features
for feat, imp in zip(selected_rfe, selector_rfe.estimator_.feature_importances_):
    print(f'  {feat:30s}  importance = {imp:.4f}')

rf_rfe = RandomForestClassifier(n_estimators=100, random_state=42)
rf_rfe.fit(X_tr[selected_rfe], y_tr)
print(f'\nTest accuracy with RFE features: {rf_rfe.score(X_te[selected_rfe], y_te):.4f}')

<a name="skip-code-17"></a>

**Expected output:** 5 selected wine features with their importances and test accuracy. RFE often selects a more principled subset than the importance threshold method because it considers feature interactions during the iterative refitting process.

<nav aria-label="Return navigation for code cell 17">
<a href="#read-code-17" aria-label="Go back and read code cell 17">&#8617; Go back and read the code cell</a>
</nav>

<a name="read-code-18"></a>

**Cell 18 — RFECV: Let Cross-Validation Choose the Feature Count**

RFECV combines RFE with cross-validation to automatically select the optimal number of features. Instead of specifying a target count, it evaluates model performance at each feature count and selects the count that maximizes cross-validated accuracy.

<nav aria-label="Code cell 18 navigation">
<a href="#skip-code-18" aria-label="Skip code cell 18 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
selector_rfecv = RFECV(
    estimator=RandomForestClassifier(n_estimators=100, random_state=42),
    step=1,
    cv=5,
    scoring='accuracy'
)
selector_rfecv.fit(X_tr, y_tr)

selected_rfecv = X_tr.columns[selector_rfecv.get_support()]
print(f'RFECV — Optimal feature count: {selector_rfecv.n_features_}')
print(f'Selected features:')
for i, f in enumerate(selected_rfecv, 1):
    print(f'  {i}. {f}')

# Plot CV score vs number of features
plt.figure(figsize=(10, 5))
plt.plot(range(1, len(selector_rfecv.cv_results_['mean_test_score']) + 1),
         selector_rfecv.cv_results_['mean_test_score'],
         marker='o', color='steelblue', linewidth=2)
plt.axvline(x=selector_rfecv.n_features_, color='red', linestyle='--', linewidth=2,
            label=f'Optimal: {selector_rfecv.n_features_} features')
plt.xlabel('Number of Features')
plt.ylabel('Cross-Validated Accuracy')
plt.title('RFECV — CV Accuracy vs Feature Count (Wine Dataset)')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('wine_rfecv.png', dpi=150, bbox_inches='tight')
plt.show()

rf_rfecv = RandomForestClassifier(n_estimators=100, random_state=42)
rf_rfecv.fit(X_tr[selected_rfecv], y_tr)
print(f'\nTest accuracy with RFECV features: {rf_rfecv.score(X_te[selected_rfecv], y_te):.4f}')

<a name="skip-code-18"></a>

**Expected output:** A plot showing how cross-validated accuracy changes as features are added, with a red line marking the optimal feature count. The curve typically rises quickly, plateaus, then may dip if noise features are added. The red line marks where the plateau is reached.

**Accessibility note:** Add a Markdown cell describing the RFECV curve: where does accuracy plateau? Does adding more features hurt, help, or make no difference?

<nav aria-label="Return navigation for code cell 18">
<a href="#read-code-18" aria-label="Go back and read code cell 18">&#8617; Go back and read the code cell</a>
</nav>

---
# Part 6: Boosted Trees on the Titanic Dataset

We started with a single tree, moved to a Random Forest (bagging), and now we will explore two **boosting** methods: **XGBoost** and **CatBoost**.

## Bagging vs Boosting

### Bagging (Random Forest)
- Trains many trees **in parallel** on different bootstrap samples
- Each tree is independent — errors are uncorrelated
- Combines predictions by **majority vote** or averaging
- Primarily reduces **variance**

### Boosting (XGBoost / CatBoost)
- Trains trees **sequentially** — each tree corrects the errors of its predecessor
- Each new tree fits the **residual errors** of the current ensemble
- Combines predictions by **weighted sum**
- Primarily reduces **bias** (with some variance reduction)

### XGBoost vs CatBoost
- **XGBoost:** Regularized gradient boosting; fast on numeric features; categorical features must be encoded manually
- **CatBoost:** Developed by Yandex; handles categorical features **natively** (no encoding needed); often strong out of the box

## The Titanic Dataset

The Titanic dataset contains passenger information from the 1912 disaster. The task is to predict whether a passenger survived (1) or did not survive (0). Features include age, sex, passenger class, fare, port of embarkation, and cabin deck.

This is an ideal dataset for boosted trees because:
- Binary classification with imbalanced classes
- Mix of numeric and categorical features (XGBoost vs CatBoost comparison is meaningful)
- Domain knowledge matters (sex, class, and age were survival predictors)
- Missing values exist in the real data

---

<a name="read-code-19"></a>

**Cell 19 — Load and Inspect the Titanic Dataset**

This cell loads the Titanic dataset from seaborn, prints the shape and column information, and shows the first 5 rows. Note which columns have missing values — this is important for preprocessing.

<nav aria-label="Code cell 19 navigation">
<a href="#skip-code-19" aria-label="Skip code cell 19 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
df_titanic = sns.load_dataset('titanic')
print(f'Shape: {df_titanic.shape}')
print(f'\nColumn info:')
print(df_titanic.info())
print(f'\nMissing values per column:')
print(df_titanic.isnull().sum()[df_titanic.isnull().sum() > 0])
df_titanic.head()

<a name="skip-code-19"></a>

**Expected output:** 891 rows, 15 columns. Key missing values: `age` (177 missing), `deck` (688 missing — mostly unusable), and `embark_town`/`embarked` (2 missing). Your preprocessing choices for `age` will affect model performance — note them in the task below.

<nav aria-label="Return navigation for code cell 19">
<a href="#read-code-19" aria-label="Go back and read code cell 19">&#8617; Go back and read the code cell</a>
</nav>

## ✏️ Your Task — Preprocessing

Edit this Markdown cell and explain the preprocessing decisions you make in Cell 20:

1. **Which columns did you drop and why?**
   *[Your answer — think about columns that are identifiers, nearly all missing, or redundant]*

2. **How did you handle missing values in `age`?**
   *[Your answer — median imputation? Mean? Drop rows? Justify your choice.]*

3. **How did you encode categorical features for XGBoost?**
   *[Your answer — LabelEncoder? OneHotEncoder? Note that CatBoost does not need encoding.]*

---
**YOUR ANSWERS ABOVE** — replace the italicized lines.

<a name="read-code-20"></a>

**Cell 20 — Preprocess Titanic Data and Train/Test Split**

This cell selects useful features, drops or imputes missing values, encodes categorical variables for XGBoost, and creates a stratified 75/25 train/test split. CatBoost receives a separate version of the data with the raw categorical columns since it handles them natively.

<nav aria-label="Code cell 20 navigation">
<a href="#skip-code-20" aria-label="Skip code cell 20 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
# Feature selection and cleaning
df_clean = df_titanic[['survived', 'pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked']].copy()

# Impute missing values
df_clean['age']      = df_clean['age'].fillna(df_clean['age'].median())
df_clean['embarked'] = df_clean['embarked'].fillna(df_clean['embarked'].mode()[0])
df_clean = df_clean.dropna()  # Remove any remaining nulls

print(f'Clean dataset shape: {df_clean.shape}')
print(f'Survival rate: {df_clean["survived"].mean():.3f}')

# Encode for XGBoost (needs numeric)
df_encoded = df_clean.copy()
df_encoded['sex']      = LabelEncoder().fit_transform(df_encoded['sex'])
df_encoded['embarked'] = LabelEncoder().fit_transform(df_encoded['embarked'])

# Features and target
feature_cols = ['pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked']
cat_cols     = ['sex', 'embarked']  # for CatBoost

X_t    = df_encoded[feature_cols]
y_t    = df_encoded['survived']
X_t_raw = df_clean[feature_cols]   # raw strings for CatBoost

X_t_tr, X_t_te, y_t_tr, y_t_te = train_test_split(
    X_t, y_t, test_size=0.25, random_state=42, stratify=y_t
)
X_t_raw_tr = X_t_raw.iloc[X_t_tr.index - X_t_tr.index.min()].reset_index(drop=True)
X_t_raw_te = X_t_raw.iloc[X_t_te.index - X_t_te.index.min()].reset_index(drop=True)

# Safer split for raw CatBoost data
X_t_raw_tr, X_t_raw_te = train_test_split(
    X_t_raw, test_size=0.25, random_state=42, stratify=y_t
)
y_t_tr2, y_t_te2 = train_test_split(y_t, test_size=0.25, random_state=42, stratify=y_t)

print(f'Train: {X_t_tr.shape}  |  Test: {X_t_te.shape}')

<a name="skip-code-20"></a>

**Expected output:** Clean dataset shape around (889, 8), survival rate around 0.38, and the train/test split sizes. The survival rate confirms class imbalance — 62% did not survive — so accuracy alone is insufficient; check precision and recall per class.

<nav aria-label="Return navigation for code cell 20">
<a href="#read-code-20" aria-label="Go back and read code cell 20">&#8617; Go back and read the code cell</a>
</nav>

## ✏️ Your Task — Build a Small, Thoughtful Parameter Grid

For each model (RandomForest, XGBoost, CatBoost):

1. Pick **2–3 hyperparameters** to tune
2. Provide **2–3 reasonable values** for each
3. Justify your choices in a few sentences before running the grid
4. Keep your total grid under **12 combinations** per model

If you use an AI assistant for inspiration, include constraints in your prompt: *'Suggest a very small parameter grid (under 12 total combinations) with only 2-3 hyperparameters and 2 choices per hyperparameter for Titanic survival prediction.'* Use AI as a consultant — not an autopilot.

<a name="read-code-21"></a>

**Cell 21 — Grid Search for Random Forest on Titanic**

Define your Random Forest parameter grid in `rf_grid`, run GridSearchCV, and apply the best parameters. Print the best CV accuracy and test accuracy.

<nav aria-label="Code cell 21 navigation">
<a href="#skip-code-21" aria-label="Skip code cell 21 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
# YOUR CODE HERE
# Define your Random Forest parameter grid
rf_grid = {
    # Add your hyperparameters and values
    # Example structure:
    # 'max_depth':         [3, 5],
    # 'min_samples_leaf':  [2, 4],
    # 'criterion':         ['gini', 'entropy']
}

# YOUR CODE HERE
# Run GridSearchCV and print best parameters and accuracy
# rf_cv = GridSearchCV(RandomForestClassifier(n_estimators=100, random_state=42),
#                      rf_grid, cv=5, scoring='accuracy', n_jobs=-1)
# rf_cv.fit(X_t_tr, y_t_tr)
# print('RF Best CV accuracy:', rf_cv.best_score_)
# print('RF Best parameters:', rf_cv.best_params_)

# YOUR CODE HERE
# Apply best parameters and evaluate on test set
# rf_titanic = RandomForestClassifier(n_estimators=100, random_state=42).set_params(**rf_cv.best_params_)
# rf_titanic.fit(X_t_tr, y_t_tr)
# rf_acc = rf_titanic.score(X_t_te, y_t_te)
# print(f'RF Test accuracy: {rf_acc:.4f}')

<a name="skip-code-21"></a>

**Expected accuracy range for Random Forest on Titanic: 0.78 to 0.84**

If your accuracy is below 0.75, check your preprocessing — specifically whether `age` was imputed and `sex` was encoded. The `sex` feature is historically the strongest predictor of Titanic survival.

<nav aria-label="Return navigation for code cell 21">
<a href="#read-code-21" aria-label="Go back and read code cell 21">&#8617; Go back and read the code cell</a>
</nav>

<a name="read-code-22"></a>

**Cell 22 — Grid Search for XGBoost on Titanic**

XGBoost has its own hyperparameter vocabulary. Key parameters to consider:
- `n_estimators`: number of boosting rounds (trees)
- `max_depth`: depth of each tree (typically 3-6 for XGBoost)
- `learning_rate`: step size shrinkage (smaller = more conservative; try 0.05-0.3)
- `subsample`: fraction of samples used per tree (similar to bootstrap)

Define your grid, run GridSearchCV, and evaluate.

<nav aria-label="Code cell 22 navigation">
<a href="#skip-code-22" aria-label="Skip code cell 22 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
# YOUR CODE HERE
# Define your XGBoost parameter grid
xgb_grid = {
    # Add your hyperparameters and values
    # 'n_estimators':  [100, 200],
    # 'max_depth':     [3, 5],
    # 'learning_rate': [0.05, 0.1]
}

# YOUR CODE HERE
# Run GridSearchCV
# xgb_cv = GridSearchCV(XGBClassifier(random_state=42, eval_metric='logloss', verbosity=0),
#                       xgb_grid, cv=5, scoring='accuracy', n_jobs=-1)
# xgb_cv.fit(X_t_tr, y_t_tr)
# print('XGB Best CV accuracy:', xgb_cv.best_score_)
# print('XGB Best parameters:', xgb_cv.best_params_)

# YOUR CODE HERE
# Apply best parameters and evaluate
# xgb_titanic = XGBClassifier(random_state=42, eval_metric='logloss', verbosity=0).set_params(**xgb_cv.best_params_)
# xgb_titanic.fit(X_t_tr, y_t_tr)
# xgb_acc = xgb_titanic.score(X_t_te, y_t_te)
# print(f'XGB Test accuracy: {xgb_acc:.4f}')

<a name="skip-code-22"></a>

**Expected accuracy range for XGBoost on Titanic: 0.78 to 0.85**

XGBoost generally matches or beats Random Forest on tabular data, especially with careful hyperparameter tuning. A smaller `learning_rate` with more `n_estimators` often outperforms a higher learning rate.

<nav aria-label="Return navigation for code cell 22">
<a href="#read-code-22" aria-label="Go back and read code cell 22">&#8617; Go back and read the code cell</a>
</nav>

<a name="read-code-23"></a>

**Cell 23 — Grid Search for CatBoost on Titanic**

CatBoost's key advantage here: it accepts raw string categorical columns (`sex` as 'male'/'female', `embarked` as 'C'/'Q'/'S') without encoding. Specify which columns are categorical using the `cat_features` parameter. Key CatBoost hyperparameters:
- `iterations`: number of boosting rounds
- `depth`: tree depth (usually 4-8)
- `learning_rate`: step size

Use `X_t_raw_tr` and `X_t_raw_te` (the unencoded versions) for CatBoost.

<nav aria-label="Code cell 23 navigation">
<a href="#skip-code-23" aria-label="Skip code cell 23 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
# Note: CatBoost uses X_t_raw_tr / X_t_raw_te (raw string categoricals)
# Specify cat_features as a list of column names
cat_feature_names = ['sex', 'embarked']

# YOUR CODE HERE
# Define your CatBoost parameter grid
cb_grid = {
    # 'iterations':    [100, 200],
    # 'depth':         [4, 6],
    # 'learning_rate': [0.05, 0.1]
}

# YOUR CODE HERE
# Run GridSearchCV
# Note: CatBoostClassifier accepts cat_features in its constructor
# cb_cv = GridSearchCV(
#     CatBoostClassifier(cat_features=cat_feature_names, verbose=0, random_state=42),
#     cb_grid, cv=5, scoring='accuracy', n_jobs=1  # n_jobs=1 for CatBoost stability
# )
# cb_cv.fit(X_t_raw_tr, y_t_tr2)
# print('CatBoost Best CV accuracy:', cb_cv.best_score_)
# print('CatBoost Best parameters:', cb_cv.best_params_)

# YOUR CODE HERE
# Apply best parameters and evaluate
# cb_titanic = CatBoostClassifier(cat_features=cat_feature_names, verbose=0, random_state=42).set_params(**cb_cv.best_params_)
# cb_titanic.fit(X_t_raw_tr, y_t_tr2)
# cb_acc = cb_titanic.score(X_t_raw_te, y_t_te2)
# print(f'CatBoost Test accuracy: {cb_acc:.4f}')

<a name="skip-code-23"></a>

**Expected accuracy range for CatBoost on Titanic: 0.79 to 0.84**

CatBoost is particularly valuable when you have many high-cardinality categorical features. For the Titanic dataset with only `sex` and `embarked` as categoricals, the advantage over XGBoost may be modest — but you avoid the risk of information loss from label encoding.

<nav aria-label="Return navigation for code cell 23">
<a href="#read-code-23" aria-label="Go back and read code cell 23">&#8617; Go back and read the code cell</a>
</nav>

## ✏️ Comparison Questions

Answer all three questions below by editing this Markdown cell.

**1. Which model performed best on the test set? By how much?**

*[Your answer — include specific accuracy numbers for all three models]*

**2. Did boosting (XGBoost / CatBoost) noticeably outperform the bagging method (Random Forest)?**

*[Your answer — discuss whether the improvement is meaningful or within noise]*

**3. How many features did your best model effectively use? Look at feature importances.**

*[Your answer — add a cell below to plot feature importances for your best model if you have not already done so]*

**4. Explain the difference between Random Forests, XGBoost, and CatBoost in your own words:**

*[Your answer — focus on the learning process: parallel vs sequential, how errors are corrected, and categorical handling]*

---
**YOUR ANSWERS ABOVE** — replace the italicized lines.

---
# Part 7: With or Without Passenger Class?

The Titanic dataset contains `pclass` (1st, 2nd, or 3rd class). This feature is strongly correlated with survival — first-class passengers had priority access to lifeboats. But it may also be a **proxy** for other features like `fare` (first class passengers paid more) or even correlated with `age` and `sex`.

**Your task:** Run two Random Forest models — one with `pclass` and one without. Compare accuracy, then answer the conceptual questions below.

---

<a name="read-code-24"></a>

**Cell 24 — Compare Random Forest With and Without `pclass`**

This cell builds a pipeline that trains two Random Forest models on the same train/test split. Model 1 uses only physical/demographic features. Model 2 adds `pclass`. Both use a OneHotEncoder for the sex and embarked columns inside a Pipeline.

<nav aria-label="Code cell 24 navigation">
<a href="#skip-code-24" aria-label="Skip code cell 24 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline

# Clean data
df_pipe = df_titanic[['survived','pclass','sex','age','sibsp','parch','fare','embarked']].copy()
df_pipe['age']      = df_pipe['age'].fillna(df_pipe['age'].median())
df_pipe['embarked'] = df_pipe['embarked'].fillna(df_pipe['embarked'].mode()[0])
df_pipe = df_pipe.dropna()

y_pipe = df_pipe['survived']

numeric_cols = ['age', 'sibsp', 'parch', 'fare']
cat_cols_pipe = ['sex', 'embarked']

features_without_pclass = numeric_cols + cat_cols_pipe
features_with_pclass    = ['pclass'] + numeric_cols + cat_cols_pipe

X_without = df_pipe[features_without_pclass]
X_with    = df_pipe[features_with_pclass]

# Shared split
X_wo_tr, X_wo_te, y_wo_tr, y_wo_te = train_test_split(
    X_without, y_pipe, test_size=0.25, random_state=42, stratify=y_pipe
)
X_wi_tr = df_pipe.loc[X_wo_tr.index, features_with_pclass]
X_wi_te = df_pipe.loc[X_wo_te.index, features_with_pclass]

# Preprocessor
preprocessor = ColumnTransformer(transformers=[
    ('cat', OneHotEncoder(handle_unknown='ignore', drop='first'), cat_cols_pipe),
    ('num', 'passthrough', numeric_cols)
])
preprocessor_with = ColumnTransformer(transformers=[
    ('cat', OneHotEncoder(handle_unknown='ignore', drop='first'), cat_cols_pipe),
    ('num', 'passthrough', ['pclass'] + numeric_cols)
])

rf_wo_pipe = Pipeline([('pre', preprocessor),      ('model', RandomForestClassifier(n_estimators=100, random_state=42))])
rf_wi_pipe = Pipeline([('pre', preprocessor_with), ('model', RandomForestClassifier(n_estimators=100, random_state=42))])

rf_wo_pipe.fit(X_wo_tr, y_wo_tr)
rf_wi_pipe.fit(X_wi_tr, y_wo_tr)

acc_without = rf_wo_pipe.score(X_wo_te, y_wo_te)
acc_with    = rf_wi_pipe.score(X_wi_te, y_wo_te)

print(f'Test accuracy (without pclass): {acc_without:.4f}')
print(f'Test accuracy (with pclass):    {acc_with:.4f}')
print(f'Difference:                     {acc_with - acc_without:+.4f}')

<a name="skip-code-24"></a>

**Expected output:** Two accuracy scores and their difference. Adding `pclass` typically improves accuracy by 2–5 percentage points on Titanic because ticket class was a direct determinant of lifeboat access — it is genuine signal, not just a proxy.

This result directly parallels the Wine dataset insight from Part 5: **feature choice and domain knowledge often matter more than the specific algorithm.**

<nav aria-label="Return navigation for code cell 24">
<a href="#read-code-24" aria-label="Go back and read code cell 24">&#8617; Go back and read the code cell</a>
</nav>

## ✏️ Should You Include `pclass`?

Answer the following questions by editing this Markdown cell.

**1. Is `pclass` biologically or causally meaningful?**

*[Your answer — was passenger class a direct cause of survival outcomes, or is it a proxy for wealth/privilege that also shows up in fare and embarkation port?]*

**2. Does `pclass` duplicate information from other features?**

*[Your answer — how strongly is pclass correlated with fare? Could the model learn the same pattern from fare alone?]*

**3. Could including `pclass` hurt generalization?**

*[Your answer — if you deployed this model to predict survival on a modern ship with different class structures, would pclass remain meaningful?]*

**4. How would RandomForest, XGBoost, and CatBoost handle `pclass` differently?**

*[Your answer — note that pclass is an ordinal variable (1 < 2 < 3). How does CatBoost treat it vs RandomForest? Does this matter?]*

---
**YOUR ANSWERS ABOVE** — replace the italicized lines.

---
# Summary

## What You Practiced

**Part 1–2 — Decision Tree Fundamentals:**
Gini Impurity and Entropy as splitting criteria, Information Gain calculation, reading tree diagrams, and understanding why certain features appear at the root.

**Part 3 — Random Forests and Hyperparameters:**
The Bias-Variance tradeoff, how each hyperparameter affects it, bagging as a variance-reduction technique, and cross-validation as the evaluation standard.

**Part 4 — Grid Search:**
Exhaustive hyperparameter optimization with cross-validation, applying best parameters to a held-out test set, and interpreting the confusion matrix.

**Part 5 — Feature Selection:**
Three methods for reducing feature dimensionality: importance threshold (fast, embedded), RFE (thorough, wrapper), and RFECV (automated, uses CV to choose the count).

**Part 6 — Boosted Trees:**
How boosting differs from bagging, XGBoost vs CatBoost, and the practical value of native categorical handling.

**Part 7 — Feature Inclusion Analysis:**
Reasoning about whether a feature should be included based on causality, redundancy, and generalization — the most important skill that separates thoughtful data scientists from people who just add every available column.

## Key Interview Concepts

- **Gini Impurity** = $1 - \sum p_k^2$. Zero = pure. For K classes, max = $1 - 1/K$.
- **Entropy** = $-\sum p_k \log_2 p_k$. Zero = pure. Binary max = 1.0.
- **Information Gain** = parent impurity minus weighted child impurity.
- **Overfitting** = high variance = tree too deep. Fix: constrain depth, use Random Forest.
- **Underfitting** = high bias = tree too shallow. Fix: relax constraints.
- **Grid Search** uses cross-validation — never the test set.
- **Bagging** trains trees in parallel on different samples (Random Forest).
- **Boosting** trains trees sequentially on residual errors (XGBoost, CatBoost).
- **Feature importance** measures how much a feature is used for splitting — it does NOT tell you the direction of the relationship.

---

**To share:** Go to **Share** in Colab → set access to **Anyone with the link can view** → copy the link → paste into Canvas.